In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import cdist
import os
import warnings
import time
from datetime import datetime
import openpyxl

# غیرفعال کردن هشدارهای غیرضروری
warnings.filterwarnings('ignore')

# ============================================================================
# بخش 1: تعریف تمام توابع تحلیل (12 تابع - 3 مجموعه × 4 تحلیل)
# ============================================================================

# =====================================================================
# مجموعه ۱: تحلیل‌های توربین (Turbine)
# =====================================================================

def analysis_turbine_1_comprehensive(file_path, output_filename):
    """تحلیل جامع توربین - DBSCAN + 3-Sigma + Partial Correlation + RCA"""
    print(f"\n{'='*60}")
    print(f"🔄 [توربین-1] تحلیل جامع")
    print(f"{'='*60}")
    
    all_features = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
                    'AssetID_9368', 'AssetID_9369', 'AssetID_9370',
                    'AssetID_9343', 'AssetID_9344', 'AssetID_9408']
    target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']
    
    try:
        df_raw = pd.read_excel(file_path)
        df_raw['date'] = pd.to_datetime(df_raw['date'])
        df_raw = df_raw.sort_values(by='date')
        print(f"✅ دیتا بارگذاری شد. تعداد: {len(df_raw):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False
    
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_raw[all_features])
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    labels = dbscan.fit_predict(scaled_data)
    
    cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}
    
    def calculate_distance(i):
        label, point = labels[i], scaled_data[i].reshape(1, -1)
        if label != -1:
            return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
        return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0
    
    df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]
    df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
    df_cleaned = df_cleaned.sort_values(by='date').set_index('date')
    
    last_date = df_cleaned.index.max()
    start_analysis_date = last_date - pd.Timedelta(days=30)
    baseline_end = start_analysis_date - pd.Timedelta(days=1)
    baseline_start = baseline_end - pd.Timedelta(days=30)
    
    df_baseline = df_cleaned[(df_cleaned.index >= baseline_start) & (df_cleaned.index <= baseline_end)]
    df_fault = df_cleaned[df_cleaned.index >= start_analysis_date].copy()
    
    def get_sigma_status(val, mean, std):
        if std == 0: return 'Normal'
        deviation = abs(val - mean) / std
        if deviation > 3: return 'Action Required'
        elif deviation > 2: return 'Warning'
        elif deviation > 1: return 'Normal (Minor Change)'
        else: return 'Normal'
    
    for col in target_sensors:
        m = df_baseline[col].mean()
        s = df_baseline[col].std() if df_baseline[col].std() != 0 else 1e-6
        df_fault[f'Status_{col}'] = df_fault[col].apply(lambda x: get_sigma_status(x, m, s))
    
    def get_partial_corr(data, columns):
        corr_matrix = data[columns].corr().values
        precision = np.linalg.inv(corr_matrix + np.eye(corr_matrix.shape[0])*1e-6)
        d = np.sqrt(np.diag(precision))
        partial_corr = -precision / np.outer(d, d)
        np.fill_diagonal(partial_corr, 1.0)
        return pd.DataFrame(partial_corr, index=columns, columns=columns)
    
    pcorr_baseline = get_partial_corr(df_baseline, all_features)
    pcorr_fault = get_partial_corr(df_fault[all_features], all_features)
    delta_pcorr = pcorr_fault - pcorr_baseline
    
    deviation_scores = {c: abs((df_fault[c].mean() - df_baseline[c].mean()) / (df_baseline[c].std() or 1e-6)) for c in all_features}
    change_scores = {c: np.mean([abs(delta_pcorr.loc[c, o]) for o in all_features if o != c]) for c in all_features}
    
    def normalize_dict(d):
        vals = np.array(list(d.values()))
        if vals.max() == vals.min(): return {k: 0.5 for k in d.keys()}
        return {k: (v - vals.min()) / (vals.max() - vals.min()) for k, v in d.items()}
    
    dev_norm = normalize_dict(deviation_scores)
    chg_norm = normalize_dict(change_scores)
    
    rca_list = []
    for c in all_features:
        strength_fault = np.mean([abs(pcorr_fault.loc[c, o]) for o in all_features if o != c])
        score = (dev_norm[c]*0.5) + (chg_norm[c]*0.5)
        rca_list.append({
            'Sensor': c, 'Change_Normalized': chg_norm[c], 'Change_Score_Delta': change_scores[c],
            'Deviation_Normalized': dev_norm[c], 'Deviation_Score': deviation_scores[c],
            'Strength_Score_Fault': strength_fault, 'RCA_Score': score,
            'Root_Cause_Suspicion': score*(1 - strength_fault)
        })
    
    df_rca_summary = pd.DataFrame(rca_list).sort_values(by='Root_Cause_Suspicion', ascending=False)
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            status_cols = [f'Status_{c}' for c in target_sensors]
            df_fault[target_sensors + status_cols].reset_index().to_excel(writer, sheet_name='Fault_Status', index=False)
            df_rca_summary.to_excel(writer, sheet_name='RCA_Summary', index=False)
            delta_pcorr.to_excel(writer, sheet_name='Delta_Correlation_Matrix')
        print(f"✅ ذخیره شد: {output_filename}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_turbine_2_ewma(file_path, output_filename):
    """تحلیل EWMA توربین - کنترل حدود + RCA"""
    print(f"\n{'='*60}")
    print(f"🔄 [توربین-2] تحلیل EWMA")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']
    LAMBDA = 0.2
    
    try:
        df = pd.read_excel(file_path, parse_dates=['date'])
        df.set_index('date', inplace=True)
        print(f"✅ دیتا بارگذاری شد. تعداد: {len(df):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False
    
    last_date = df.index.max()
    split_date = last_date - pd.Timedelta(days=30)
    baseline_start = split_date - pd.Timedelta(days=30)
    
    df_baseline = df.loc[baseline_start:split_date].copy()
    df_fault = df.loc[split_date:last_date].copy()
    
    def calculate_ewma(data, lambda_val):
        ewma_values = np.zeros(len(data))
        if len(data) == 0: return ewma_values
        ewma_values[0] = data[0]
        for t in range(1, len(data)):
            ewma_values[t] = lambda_val * data[t] + (1 - lambda_val) * ewma_values[t-1]
        return ewma_values
    
    def calculate_control_limits(mean, std, lambda_val):
        factor = 3 * std * np.sqrt(lambda_val / (2 - lambda_val))
        return mean + factor, mean - factor
    
    target_analysis_results = []
    for col in target_sensors:
        if col not in df_fault.columns or col not in df_baseline.columns:
            continue
        mean_base = df_baseline[col].mean()
        std_base = df_baseline[col].std()
        ucl, lcl = calculate_control_limits(mean_base, std_base, LAMBDA)
        fault_data = df_fault[col].values
        ewma_values = calculate_ewma(fault_data, LAMBDA)
        current_ewma = ewma_values[-1]
        
        time_diff_series = df_fault.index.to_series().diff().dt.total_seconds() / 3600.0
        val_diff = df_fault[col].diff()
        instant_slopes = val_diff / time_diff_series
        avg_slope = instant_slopes.mean()
        current_raw_value = df_fault[col].iloc[-1]
        
        if avg_slope <= 0 or current_raw_value >= ucl:
            hours_to_ucl = 0.0
        else:
            hours_to_ucl = round((ucl - current_raw_value) / avg_slope, 2)
        
        if np.isnan(hours_to_ucl) or np.isinf(hours_to_ucl):
            hours_to_ucl = 0.0
        
        out_of_control = "Yes ⚠️" if (current_ewma > ucl or current_ewma < lcl) else "No"
        
        target_analysis_results.append({
            'Sensor': col, 'Current_Raw_Value': round(current_raw_value, 4),
            'Current_EWMA': round(current_ewma, 4), 'UCL': round(ucl, 4),
            'LCL': round(lcl, 4), 'Baseline_Mean': round(mean_base, 4),
            'Baseline_Std': round(std_base, 4),
            'Average_Slope_per_Hour': round(avg_slope, 6),
            'Hours_to_UCL': hours_to_ucl, 'Out_Of_Control': out_of_control
        })
    
    # RCA Ranking
    numeric_cols = df_baseline.select_dtypes(include=[np.number]).columns.tolist()
    rca_list = []
    for c in numeric_cols:
        if c not in df_fault.columns: continue
        mean_base = df_baseline[c].mean()
        std_base = df_baseline[c].std()
        mean_fault = df_fault[c].mean()
        deviation_score = abs(mean_fault - mean_base) / std_base if std_base > 0 else 0
        rca_list.append({'Sensor': c, 'Baseline_Mean': round(mean_base, 4),
                        'Fault_Mean': round(mean_fault, 4),
                        'Deviation_Score': round(deviation_score, 4)})
    rca_summary = pd.DataFrame(rca_list).sort_values('Deviation_Score', ascending=False)
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            pd.DataFrame(target_analysis_results).to_excel(writer, sheet_name='Speed_Analysis', index=False)
            rca_summary.to_excel(writer, sheet_name='RCA_Ranking', index=False)
            pd.DataFrame({
                'Parameter': ['Lambda (λ)', 'UCL Formula', 'LCL Formula', 'EWMA Formula', 'Hours_to_UCL Rule'],
                'Value': [f'{LAMBDA}', 'μ + 3σ√(λ/(2-λ))', 'μ - 3σ√(λ/(2-λ))',
                         'E_t = λ·X_t + (1-λ)·E_{t-1}',
                         'If slope>0 and value<UCL → (UCL-value)/slope | Else → 0']
            }).to_excel(writer, sheet_name='Methodology', index=False)
        print(f"✅ ذخیره شد: {output_filename}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_turbine_3_ewma_dbscan(file_path, output_filename):
    """تحلیل EWMA توربین با پاکسازی DBSCAN"""
    print(f"\n{'='*60}")
    print(f"🔄 [توربین-3] EWMA با DBSCAN")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']
    
    try:
        df = pd.read_excel(file_path, parse_dates=['date'])
        df = df.sort_values('date')
        print(f"✅ دیتا بارگذاری شد. تعداد: {len(df):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False
    
    last_date = df['date'].max()
    fault_start = last_date - pd.Timedelta(days=30)
    baseline_start = fault_start - pd.Timedelta(days=30)
    
    df_baseline = df[(df['date'] >= baseline_start) & (df['date'] < fault_start)].copy()
    df_fault = df[df['date'] >= fault_start].copy()
    
    def remove_outliers_dbscan(series):
        if series.empty or len(series) < 5: return series
        X = series.values.reshape(-1, 1)
        std_val = series.std()
        eps = 0.1 if (pd.isna(std_val) or std_val == 0) else max(std_val * 0.5, 0.01)
        min_samples = max(min(5, len(series)-1), 1)
        clusters = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X)
        return series if (clusters == -1).all() else series[clusters != -1]
    
    ewma_list = []
    baseline_stats = []
    
    for col in target_sensors:
        if col not in df.columns: continue
        temp_fault = df_fault[['date', col]].copy()
        temp_fault = temp_fault.rename(columns={col: 'Value'})
        temp_fault['AssetID'] = col
        temp_fault['EWMA'] = temp_fault['Value'].ewm(alpha=0.2, adjust=False).mean()
        ewma_list.append(temp_fault[['date', 'AssetID', 'Value', 'EWMA']])
        
        if col in df_baseline.columns:
            clean_baseline = remove_outliers_dbscan(df_baseline[col].dropna())
            if not clean_baseline.empty:
                mean_val, std_val = clean_baseline.mean(), clean_baseline.std()
                baseline_stats.append({
                    'AssetID': col, 'Clean_Mean': round(mean_val, 4),
                    'Clean_Std': round(std_val, 4),
                    'Normal_Upper_Limit(3Sigma)': round(mean_val + 3*std_val, 4),
                    'Outliers_Removed': len(df_baseline[col]) - len(clean_baseline)
                })
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            if ewma_list:
                pd.concat(ewma_list, ignore_index=True).to_excel(writer, sheet_name='EWMA_Comparison', index=False)
            if baseline_stats:
                pd.DataFrame(baseline_stats).to_excel(writer, sheet_name='Baseline_Stats', index=False)
        print(f"✅ ذخیره شد: {output_filename}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_turbine_4_dual_ewma(file_path, output_filename):
    """تحلیل Dual EWMA توربین (روزانه/هفتگی)"""
    print(f"\n{'='*60}")
    print(f"🔄 [توربین-4] Dual EWMA")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']
    
    try:
        df = pd.read_excel(file_path, parse_dates=['date'])
        df = df.sort_values('date')
        print(f"✅ دیتا بارگذاری شد. تعداد: {len(df):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False
    
    last_date = df['date'].max()
    one_month_ago = last_date - pd.Timedelta(days=30)
    df_recent = df[df['date'] >= one_month_ago].copy()
    
    alpha_fast, alpha_slow = 0.22, 0.035
    results_list = []
    
    for col in target_sensors:
        if col in df_recent.columns:
            temp_df = df_recent[['date', col]].copy()
            temp_df = temp_df.rename(columns={col: 'Raw_Value'})
            temp_df['AssetID'] = col
            temp_df['Daily_EWMA_Fast'] = temp_df['Raw_Value'].ewm(alpha=alpha_fast, adjust=False).mean()
            temp_df['Weekly_EWMA_Slow'] = temp_df['Raw_Value'].ewm(alpha=alpha_slow, adjust=False).mean()
            temp_df['Signal_Gap'] = temp_df['Daily_EWMA_Fast'] - temp_df['Weekly_EWMA_Slow']
            temp_df['Deviation_Percent'] = (temp_df['Signal_Gap'] / temp_df['Weekly_EWMA_Slow']) * 100
            results_list.append(temp_df[['date', 'AssetID', 'Raw_Value', 'Daily_EWMA_Fast',
                                        'Weekly_EWMA_Slow', 'Signal_Gap', 'Deviation_Percent']])
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            if results_list:
                pd.concat(results_list, ignore_index=True).to_excel(writer, sheet_name='Maintenance_Strategy', index=False)
        print(f"✅ ذخیره شد: {output_filename}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


# =====================================================================
# مجموعه ۲: تحلیل‌های ژنراتور (Generator)
# =====================================================================

def analysis_generator_1_comprehensive(file_path, output_filename):
    """تحلیل جامع ژنراتور - DBSCAN + 3-Sigma + Partial Correlation + RCA"""
    print(f"\n{'='*60}")
    print(f"🔄 [ژنراتور-1] تحلیل جامع")
    print(f"{'='*60}")
    
    all_features = ['AssetID_9343','AssetID_9344', 'AssetID_9357', 'AssetID_9362', 'AssetID_9363', 
                    'AssetID_9364', 'AssetID_9365', 'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 
                    'AssetID_9372', 'AssetID_9373']
    target_sensors = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 
                      'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
    
    try:
        df_raw = pd.read_excel(file_path)
        df_raw['date'] = pd.to_datetime(df_raw['date'])
        df_raw = df_raw.sort_values(by='date')
        print(f"✅ دیتا بارگذاری شد. تعداد: {len(df_raw):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False
    
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_raw[all_features])
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    labels = dbscan.fit_predict(scaled_data)
    
    cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}
    
    def calculate_distance(i):
        label, point = labels[i], scaled_data[i].reshape(1, -1)
        if label != -1:
            return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
        return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0
    
    df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]
    df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
    df_cleaned = df_cleaned.sort_values(by='date').set_index('date')
    
    last_date = df_cleaned.index.max()
    start_analysis_date = last_date - pd.Timedelta(days=30)
    baseline_end = start_analysis_date - pd.Timedelta(days=1)
    baseline_start = baseline_end - pd.Timedelta(days=30)
    
    df_baseline = df_cleaned[(df_cleaned.index >= baseline_start) & (df_cleaned.index <= baseline_end)]
    df_fault = df_cleaned[df_cleaned.index >= start_analysis_date].copy()
    
    def get_sigma_status(val, mean, std):
        if std == 0: return 'Normal'
        deviation = abs(val - mean) / std
        if deviation > 3: return 'Action Required'
        elif deviation > 2: return 'Warning'
        elif deviation > 1: return 'Normal (Minor Change)'
        else: return 'Normal'
    
    for col in target_sensors:
        m = df_baseline[col].mean()
        s = df_baseline[col].std() if df_baseline[col].std() != 0 else 1e-6
        df_fault[f'Status_{col}'] = df_fault[col].apply(lambda x: get_sigma_status(x, m, s))
    
    def get_partial_corr(data, columns):
        corr_matrix = data[columns].corr().values
        precision = np.linalg.inv(corr_matrix + np.eye(corr_matrix.shape[0])*1e-6)
        d = np.sqrt(np.diag(precision))
        partial_corr = -precision / np.outer(d, d)
        np.fill_diagonal(partial_corr, 1.0)
        return pd.DataFrame(partial_corr, index=columns, columns=columns)
    
    pcorr_baseline = get_partial_corr(df_baseline, all_features)
    pcorr_fault = get_partial_corr(df_fault[all_features], all_features)
    delta_pcorr = pcorr_fault - pcorr_baseline
    
    deviation_scores = {c: abs((df_fault[c].mean() - df_baseline[c].mean()) / (df_baseline[c].std() or 1e-6)) for c in all_features}
    change_scores = {c: np.mean([abs(delta_pcorr.loc[c, o]) for o in all_features if o != c]) for c in all_features}
    
    def normalize_dict(d):
        vals = np.array(list(d.values()))
        if vals.max() == vals.min(): return {k: 0.5 for k in d.keys()}
        return {k: (v - vals.min()) / (vals.max() - vals.min()) for k, v in d.items()}
    
    dev_norm = normalize_dict(deviation_scores)
    chg_norm = normalize_dict(change_scores)
    
    rca_list = []
    for c in all_features:
        strength_fault = np.mean([abs(pcorr_fault.loc[c, o]) for o in all_features if o != c])
        score = (dev_norm[c]*0.5) + (chg_norm[c]*0.5)
        rca_list.append({
            'Sensor': c, 'Change_Normalized': chg_norm[c], 'Change_Score_Delta': change_scores[c],
            'Deviation_Normalized': dev_norm[c], 'Deviation_Score': deviation_scores[c],
            'Strength_Score_Fault': strength_fault, 'RCA_Score': score,
            'Root_Cause_Suspicion': score*(1 - strength_fault)
        })
    
    df_rca_summary = pd.DataFrame(rca_list).sort_values(by='Root_Cause_Suspicion', ascending=False)
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            status_cols = [f'Status_{c}' for c in target_sensors]
            df_fault[target_sensors + status_cols].reset_index().to_excel(writer, sheet_name='Fault_Status', index=False)
            df_rca_summary.to_excel(writer, sheet_name='RCA_Summary', index=False)
            delta_pcorr.to_excel(writer, sheet_name='Delta_Correlation_Matrix')
        print(f"✅ ذخیره شد: {output_filename}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_generator_2_ewma(file_path, output_filename):
    """تحلیل EWMA ژنراتور - کنترل حدود + RCA"""
    print(f"\n{'='*60}")
    print(f"🔄 [ژنراتور-2] تحلیل EWMA")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 
                      'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
    LAMBDA = 0.2
    
    try:
        df = pd.read_excel(file_path, parse_dates=['date'])
        df.set_index('date', inplace=True)
        print(f"✅ دیتا بارگذاری شد. تعداد: {len(df):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False
    
    last_date = df.index.max()
    split_date = last_date - pd.Timedelta(days=30)
    baseline_start = split_date - pd.Timedelta(days=30)
    
    df_baseline = df.loc[baseline_start:split_date].copy()
    df_fault = df.loc[split_date:last_date].copy()
    
    def calculate_ewma(data, lambda_val):
        ewma_values = np.zeros(len(data))
        if len(data) == 0: return ewma_values
        ewma_values[0] = data[0]
        for t in range(1, len(data)):
            ewma_values[t] = lambda_val * data[t] + (1 - lambda_val) * ewma_values[t-1]
        return ewma_values
    
    def calculate_control_limits(mean, std, lambda_val):
        factor = 3 * std * np.sqrt(lambda_val / (2 - lambda_val))
        return mean + factor, mean - factor
    
    target_analysis_results = []
    for col in target_sensors:
        if col not in df_fault.columns or col not in df_baseline.columns:
            continue
        mean_base = df_baseline[col].mean()
        std_base = df_baseline[col].std()
        ucl, lcl = calculate_control_limits(mean_base, std_base, LAMBDA)
        fault_data = df_fault[col].values
        ewma_values = calculate_ewma(fault_data, LAMBDA)
        current_ewma = ewma_values[-1]
        
        time_diff_series = df_fault.index.to_series().diff().dt.total_seconds() / 3600.0
        val_diff = df_fault[col].diff()
        instant_slopes = val_diff / time_diff_series
        avg_slope = instant_slopes.mean()
        current_raw_value = df_fault[col].iloc[-1]
        
        if avg_slope <= 0 or current_raw_value >= ucl:
            hours_to_ucl = 0.0
        else:
            hours_to_ucl = round((ucl - current_raw_value) / avg_slope, 2)
        
        if np.isnan(hours_to_ucl) or np.isinf(hours_to_ucl):
            hours_to_ucl = 0.0
        
        out_of_control = "Yes ⚠️" if (current_ewma > ucl or current_ewma < lcl) else "No"
        
        target_analysis_results.append({
            'Sensor': col, 'Current_Raw_Value': round(current_raw_value, 4),
            'Current_EWMA': round(current_ewma, 4), 'UCL': round(ucl, 4),
            'LCL': round(lcl, 4), 'Baseline_Mean': round(mean_base, 4),
            'Baseline_Std': round(std_base, 4),
            'Average_Slope_per_Hour': round(avg_slope, 6),
            'Hours_to_UCL': hours_to_ucl, 'Out_Of_Control': out_of_control
        })
    
    numeric_cols = df_baseline.select_dtypes(include=[np.number]).columns.tolist()
    rca_list = []
    for c in numeric_cols:
        if c not in df_fault.columns: continue
        mean_base = df_baseline[c].mean()
        std_base = df_baseline[c].std()
        mean_fault = df_fault[c].mean()
        deviation_score = abs(mean_fault - mean_base) / std_base if std_base > 0 else 0
        rca_list.append({'Sensor': c, 'Baseline_Mean': round(mean_base, 4),
                        'Fault_Mean': round(mean_fault, 4),
                        'Deviation_Score': round(deviation_score, 4)})
    rca_summary = pd.DataFrame(rca_list).sort_values('Deviation_Score', ascending=False)
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            pd.DataFrame(target_analysis_results).to_excel(writer, sheet_name='Speed_Analysis', index=False)
            rca_summary.to_excel(writer, sheet_name='RCA_Ranking', index=False)
            pd.DataFrame({
                'Parameter': ['Lambda (λ)', 'UCL Formula', 'LCL Formula', 'EWMA Formula', 'Hours_to_UCL Rule'],
                'Value': [f'{LAMBDA}', 'μ + 3σ√(λ/(2-λ))', 'μ - 3σ√(λ/(2-λ))',
                         'E_t = λ·X_t + (1-λ)·E_{t-1}',
                         'If slope>0 and value<UCL → (UCL-value)/slope | Else → 0']
            }).to_excel(writer, sheet_name='Methodology', index=False)
        print(f"✅ ذخیره شد: {output_filename}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_generator_3_ewma_dbscan(file_path, output_filename):
    """تحلیل EWMA ژنراتور با پاکسازی DBSCAN"""
    print(f"\n{'='*60}")
    print(f"🔄 [ژنراتور-3] EWMA با DBSCAN")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 
                      'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
    
    try:
        df = pd.read_excel(file_path, parse_dates=['date'])
        df = df.sort_values('date')
        print(f"✅ دیتا بارگذاری شد. تعداد: {len(df):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False
    
    last_date = df['date'].max()
    fault_start = last_date - pd.Timedelta(days=30)
    baseline_start = fault_start - pd.Timedelta(days=30)
    
    df_baseline = df[(df['date'] >= baseline_start) & (df['date'] < fault_start)].copy()
    df_fault = df[df['date'] >= fault_start].copy()
    
    def remove_outliers_dbscan(series):
        if series.empty or len(series) < 5: return series
        X = series.values.reshape(-1, 1)
        std_val = series.std()
        eps = 0.1 if (pd.isna(std_val) or std_val == 0) else max(std_val * 0.5, 0.01)
        min_samples = max(min(5, len(series)-1), 1)
        clusters = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X)
        return series if (clusters == -1).all() else series[clusters != -1]
    
    sensor_stats = {}
    for col in target_sensors:
        if col not in df.columns or col not in df_baseline.columns: continue
        clean_baseline = remove_outliers_dbscan(df_baseline[col].dropna())
        if not clean_baseline.empty:
            mean_val, std_val = clean_baseline.mean(), clean_baseline.std()
            sensor_stats[col] = {'mean': round(mean_val, 4), 'std': round(std_val, 4),
                                'upper_band': round(mean_val + 3*std_val, 4),
                                'lower_band': round(mean_val - 3*std_val, 4)}
    
    ewma_list, baseline_stats = [], []
    for col in target_sensors:
        if col not in df.columns: continue
        temp_fault = df_fault[['date', col]].copy()
        if not temp_fault.empty:
            temp_fault = temp_fault.rename(columns={col: 'Value'})
            temp_fault['AssetID'] = col
            temp_fault['EWMA'] = temp_fault['Value'].ewm(alpha=0.2, adjust=False).mean()
            if col in sensor_stats:
                temp_fault['Repeated_Mean'] = sensor_stats[col]['mean']
                temp_fault['Repeated_Std'] = sensor_stats[col]['std']
                temp_fault['upper_band'] = sensor_stats[col]['upper_band']
                temp_fault['lower_band'] = sensor_stats[col]['lower_band']
            else:
                for f in ['Repeated_Mean', 'Repeated_Std', 'upper_band', 'lower_band']:
                    temp_fault[f] = np.nan
            ewma_list.append(temp_fault[['date', 'AssetID', 'Value', 'EWMA', 
                                        'Repeated_Mean', 'Repeated_Std', 'upper_band', 'lower_band']])
        
        if col in df_baseline.columns:
            clean_baseline = remove_outliers_dbscan(df_baseline[col].dropna())
            if not clean_baseline.empty:
                mean_val, std_val = clean_baseline.mean(), clean_baseline.std()
                baseline_stats.append({
                    'AssetID': col, 'Clean_Mean': round(mean_val, 4),
                    'Clean_Std': round(std_val, 4),
                    'Normal_Upper_Limit(3Sigma)': round(mean_val + 3*std_val, 4),
                    'Normal_Lower_Limit(3Sigma)': round(mean_val - 3*std_val, 4),
                    'Outliers_Removed': len(df_baseline[col]) - len(clean_baseline),
                    'Baseline_Data_Count': len(clean_baseline),
                    'Original_Data_Count': len(df_baseline[col].dropna())
                })
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            if ewma_list:
                pd.concat(ewma_list, ignore_index=True).to_excel(writer, sheet_name='EWMA_Comparison', index=False)
            if baseline_stats:
                pd.DataFrame(baseline_stats).to_excel(writer, sheet_name='Baseline_Stats', index=False)
        print(f"✅ ذخیره شد: {output_filename}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_generator_4_dual_ewma(file_path, output_filename):
    """تحلیل Dual EWMA ژنراتور (روزانه/هفتگی)"""
    print(f"\n{'='*60}")
    print(f"🔄 [ژنراتور-4] Dual EWMA")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 
                      'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
    
    try:
        df = pd.read_excel(file_path, parse_dates=['date'])
        df = df.sort_values('date')
        print(f"✅ دیتا بارگذاری شد. تعداد: {len(df):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False
    
    last_date = df['date'].max()
    one_month_ago = last_date - pd.Timedelta(days=30)
    df_recent = df[df['date'] >= one_month_ago].copy()
    
    alpha_fast, alpha_slow = 0.22, 0.035
    results_list = []
    
    for col in target_sensors:
        if col in df_recent.columns:
            temp_df = df_recent[['date', col]].copy()
            temp_df = temp_df.rename(columns={col: 'Raw_Value'})
            temp_df['AssetID'] = col
            temp_df['Daily_EWMA_Fast'] = temp_df['Raw_Value'].ewm(alpha=alpha_fast, adjust=False).mean()
            temp_df['Weekly_EWMA_Slow'] = temp_df['Raw_Value'].ewm(alpha=alpha_slow, adjust=False).mean()
            temp_df['Signal_Gap'] = temp_df['Daily_EWMA_Fast'] - temp_df['Weekly_EWMA_Slow']
            temp_df['Deviation_Percent'] = (temp_df['Signal_Gap'] / temp_df['Weekly_EWMA_Slow']) * 100
            results_list.append(temp_df[['date', 'AssetID', 'Raw_Value', 'Daily_EWMA_Fast',
                                        'Weekly_EWMA_Slow', 'Signal_Gap', 'Deviation_Percent']])
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            if results_list:
                pd.concat(results_list, ignore_index=True).to_excel(writer, sheet_name='Maintenance_Strategy', index=False)
        print(f"✅ ذخیره شد: {output_filename}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


# =====================================================================
# مجموعه ۳: تحلیل‌های سیستم روغن‌کاری (Lubrication)
# =====================================================================

def analysis_lubrication_1_comprehensive(file_path, output_filename):
    """تحلیل جامع سیستم روغن‌کاری - DBSCAN + 3-Sigma + Partial Correlation + RCA"""
    print(f"\n{'='*60}")
    print(f"🔄 [روغن‌کاری-1] تحلیل جامع")
    print(f"{'='*60}")
    
    all_features = ['AssetID_9343','AssetID_9375', 'AssetID_8341', 'AssetID_8342', 'AssetID_8343', 
                    'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287']
    target_sensors = ['AssetID_9343','AssetID_9357','AssetID_9375', 'AssetID_8341', 'AssetID_8343', 
                      'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287']
    
    try:
        df_raw = pd.read_excel(file_path)
        df_raw['date'] = pd.to_datetime(df_raw['date'])
        df_raw = df_raw.sort_values(by='date')
        print(f"✅ دیتا بارگذاری شد. تعداد: {len(df_raw):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False
    
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_raw[all_features])
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    labels = dbscan.fit_predict(scaled_data)
    
    cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}
    
    def calculate_distance(i):
        label, point = labels[i], scaled_data[i].reshape(1, -1)
        if label != -1:
            return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
        return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0
    
    df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]
    df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
    df_cleaned = df_cleaned.sort_values(by='date').set_index('date')
    
    last_date = df_cleaned.index.max()
    start_analysis_date = last_date - pd.Timedelta(days=30)
    baseline_end = start_analysis_date - pd.Timedelta(days=1)
    baseline_start = baseline_end - pd.Timedelta(days=30)
    
    df_baseline = df_cleaned[(df_cleaned.index >= baseline_start) & (df_cleaned.index <= baseline_end)]
    df_fault = df_cleaned[df_cleaned.index >= start_analysis_date].copy()
    
    def get_sigma_status(val, mean, std):
        if std == 0: return 'Normal'
        deviation = abs(val - mean) / std
        if deviation > 3: return 'Action Required'
        elif deviation > 2: return 'Warning'
        elif deviation > 1: return 'Normal (Minor Change)'
        else: return 'Normal'
    
    for col in target_sensors:
        if col in df_baseline.columns:
            m = df_baseline[col].mean()
            s = df_baseline[col].std() if df_baseline[col].std() != 0 else 1e-6
            df_fault[f'Status_{col}'] = df_fault[col].apply(lambda x: get_sigma_status(x, m, s))
    
    def get_partial_corr(data, columns):
        corr_matrix = data[columns].corr().values
        precision = np.linalg.inv(corr_matrix + np.eye(corr_matrix.shape[0])*1e-6)
        d = np.sqrt(np.diag(precision))
        partial_corr = -precision / np.outer(d, d)
        np.fill_diagonal(partial_corr, 1.0)
        return pd.DataFrame(partial_corr, index=columns, columns=columns)
    
    pcorr_baseline = get_partial_corr(df_baseline, all_features)
    pcorr_fault = get_partial_corr(df_fault[all_features], all_features)
    delta_pcorr = pcorr_fault - pcorr_baseline
    
    deviation_scores = {c: abs((df_fault[c].mean() - df_baseline[c].mean()) / (df_baseline[c].std() or 1e-6)) for c in all_features}
    change_scores = {c: np.mean([abs(delta_pcorr.loc[c, o]) for o in all_features if o != c]) for c in all_features}
    
    def normalize_dict(d):
        vals = np.array(list(d.values()))
        if vals.max() == vals.min(): return {k: 0.5 for k in d.keys()}
        return {k: (v - vals.min()) / (vals.max() - vals.min()) for k, v in d.items()}
    
    dev_norm = normalize_dict(deviation_scores)
    chg_norm = normalize_dict(change_scores)
    
    rca_list = []
    for c in all_features:
        strength_fault = np.mean([abs(pcorr_fault.loc[c, o]) for o in all_features if o != c])
        score = (dev_norm[c]*0.5) + (chg_norm[c]*0.5)
        rca_list.append({
            'Sensor': c, 'Change_Normalized': chg_norm[c], 'Change_Score_Delta': change_scores[c],
            'Deviation_Normalized': dev_norm[c], 'Deviation_Score': deviation_scores[c],
            'Strength_Score_Fault': strength_fault, 'RCA_Score': score,
            'Root_Cause_Suspicion': score*(1 - strength_fault)
        })
    
    df_rca_summary = pd.DataFrame(rca_list).sort_values(by='Root_Cause_Suspicion', ascending=False)
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            status_cols = [f'Status_{c}' for c in target_sensors if f'Status_{c}' in df_fault.columns]
            cols_to_save = [c for c in target_sensors if c in df_fault.columns] + status_cols
            if cols_to_save:
                df_fault[cols_to_save].reset_index().to_excel(writer, sheet_name='Fault_Status', index=False)
            df_rca_summary.to_excel(writer, sheet_name='RCA_Summary', index=False)
            delta_pcorr.to_excel(writer, sheet_name='Delta_Correlation_Matrix')
        print(f"✅ ذخیره شد: {output_filename}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_lubrication_2_ewma(file_path, output_filename):
    """تحلیل EWMA سیستم روغن‌کاری - کنترل حدود + RCA"""
    print(f"\n{'='*60}")
    print(f"🔄 [روغن‌کاری-2] تحلیل EWMA")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9375', 'AssetID_8341', 'AssetID_8343', 'AssetID_8344', 
                      'AssetID_8346', 'AssetID_9286', 'AssetID_9287']
    LAMBDA = 0.2
    
    try:
        df = pd.read_excel(file_path, parse_dates=['date'])
        df.set_index('date', inplace=True)
        print(f"✅ دیتا بارگذاری شد. تعداد: {len(df):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False
    
    last_date = df.index.max()
    split_date = last_date - pd.Timedelta(days=30)
    baseline_start = split_date - pd.Timedelta(days=30)
    
    df_baseline = df.loc[baseline_start:split_date].copy()
    df_fault = df.loc[split_date:last_date].copy()
    
    def calculate_ewma(data, lambda_val):
        ewma_values = np.zeros(len(data))
        if len(data) == 0: return ewma_values
        ewma_values[0] = data[0]
        for t in range(1, len(data)):
            ewma_values[t] = lambda_val * data[t] + (1 - lambda_val) * ewma_values[t-1]
        return ewma_values
    
    def calculate_control_limits(mean, std, lambda_val):
        factor = 3 * std * np.sqrt(lambda_val / (2 - lambda_val))
        return mean + factor, mean - factor
    
    target_analysis_results = []
    for col in target_sensors:
        if col not in df_fault.columns or col not in df_baseline.columns:
            continue
        mean_base = df_baseline[col].mean()
        std_base = df_baseline[col].std()
        ucl, lcl = calculate_control_limits(mean_base, std_base, LAMBDA)
        fault_data = df_fault[col].values
        ewma_values = calculate_ewma(fault_data, LAMBDA)
        current_ewma = ewma_values[-1]
        
        time_diff_series = df_fault.index.to_series().diff().dt.total_seconds() / 3600.0
        val_diff = df_fault[col].diff()
        instant_slopes = val_diff / time_diff_series
        avg_slope = instant_slopes.mean()
        current_raw_value = df_fault[col].iloc[-1]
        
        if avg_slope <= 0 or current_raw_value >= ucl:
            hours_to_ucl = 0.0
        else:
            hours_to_ucl = round((ucl - current_raw_value) / avg_slope, 2)
        
        if np.isnan(hours_to_ucl) or np.isinf(hours_to_ucl):
            hours_to_ucl = 0.0
        
        out_of_control = "Yes ⚠️" if (current_ewma > ucl or current_ewma < lcl) else "No"
        
        target_analysis_results.append({
            'Sensor': col, 'Current_Raw_Value': round(current_raw_value, 4),
            'Current_EWMA': round(current_ewma, 4), 'UCL': round(ucl, 4),
            'LCL': round(lcl, 4), 'Baseline_Mean': round(mean_base, 4),
            'Baseline_Std': round(std_base, 4),
            'Average_Slope_per_Hour': round(avg_slope, 6),
            'Hours_to_UCL': hours_to_ucl, 'Out_Of_Control': out_of_control
        })
    
    numeric_cols = df_baseline.select_dtypes(include=[np.number]).columns.tolist()
    rca_list = []
    for c in numeric_cols:
        if c not in df_fault.columns: continue
        mean_base = df_baseline[c].mean()
        std_base = df_baseline[c].std()
        mean_fault = df_fault[c].mean()
        deviation_score = abs(mean_fault - mean_base) / std_base if std_base > 0 else 0
        rca_list.append({'Sensor': c, 'Baseline_Mean': round(mean_base, 4),
                        'Fault_Mean': round(mean_fault, 4),
                        'Deviation_Score': round(deviation_score, 4)})
    rca_summary = pd.DataFrame(rca_list).sort_values('Deviation_Score', ascending=False)
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            pd.DataFrame(target_analysis_results).to_excel(writer, sheet_name='Speed_Analysis', index=False)
            rca_summary.to_excel(writer, sheet_name='RCA_Ranking', index=False)
            pd.DataFrame({
                'Parameter': ['Lambda (λ)', 'UCL Formula', 'LCL Formula', 'EWMA Formula', 'Hours_to_UCL Rule'],
                'Value': [f'{LAMBDA}', 'μ + 3σ√(λ/(2-λ))', 'μ - 3σ√(λ/(2-λ))',
                         'E_t = λ·X_t + (1-λ)·E_{t-1}',
                         'If slope>0 and value<UCL → (UCL-value)/slope | Else → 0']
            }).to_excel(writer, sheet_name='Methodology', index=False)
        print(f"✅ ذخیره شد: {output_filename}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_lubrication_3_ewma_dbscan(file_path, output_filename):
    """تحلیل EWMA سیستم روغن‌کاری با پاکسازی DBSCAN"""
    print(f"\n{'='*60}")
    print(f"🔄 [روغن‌کاری-3] EWMA با DBSCAN")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9375', 'AssetID_8341', 'AssetID_8343', 'AssetID_8344', 
                      'AssetID_8346', 'AssetID_9286', 'AssetID_9287']
    
    try:
        df = pd.read_excel(file_path, parse_dates=['date'])
        df = df.sort_values('date')
        print(f"✅ دیتا بارگذاری شد. تعداد: {len(df):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False
    
    last_date = df['date'].max()
    fault_start = last_date - pd.Timedelta(days=30)
    baseline_start = fault_start - pd.Timedelta(days=30)
    
    df_baseline = df[(df['date'] >= baseline_start) & (df['date'] < fault_start)].copy()
    df_fault = df[df['date'] >= fault_start].copy()
    
    def remove_outliers_dbscan(series):
        if series.empty or len(series) < 5: return series
        X = series.values.reshape(-1, 1)
        std_val = series.std()
        eps = 0.1 if (pd.isna(std_val) or std_val == 0) else max(std_val * 0.5, 0.01)
        min_samples = max(min(5, len(series)-1), 1)
        clusters = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X)
        return series if (clusters == -1).all() else series[clusters != -1]
    
    sensor_stats = {}
    for col in target_sensors:
        if col not in df.columns or col not in df_baseline.columns: continue
        clean_baseline = remove_outliers_dbscan(df_baseline[col].dropna())
        if not clean_baseline.empty:
            mean_val, std_val = clean_baseline.mean(), clean_baseline.std()
            sensor_stats[col] = {'mean': round(mean_val, 4), 'std': round(std_val, 4),
                                'upper_band': round(mean_val + 3*std_val, 4),
                                'lower_band': round(mean_val - 3*std_val, 4)}
    
    ewma_list, baseline_stats = [], []
    for col in target_sensors:
        if col not in df.columns: continue
        temp_fault = df_fault[['date', col]].copy()
        if not temp_fault.empty:
            temp_fault = temp_fault.rename(columns={col: 'Value'})
            temp_fault['AssetID'] = col
            temp_fault['EWMA'] = temp_fault['Value'].ewm(alpha=0.2, adjust=False).mean()
            if col in sensor_stats:
                temp_fault['Repeated_Mean'] = sensor_stats[col]['mean']
                temp_fault['Repeated_Std'] = sensor_stats[col]['std']
                temp_fault['upper_band'] = sensor_stats[col]['upper_band']
                temp_fault['lower_band'] = sensor_stats[col]['lower_band']
            else:
                for f in ['Repeated_Mean', 'Repeated_Std', 'upper_band', 'lower_band']:
                    temp_fault[f] = np.nan
            ewma_list.append(temp_fault[['date', 'AssetID', 'Value', 'EWMA', 
                                        'Repeated_Mean', 'Repeated_Std', 'upper_band', 'lower_band']])
        
        if col in df_baseline.columns:
            clean_baseline = remove_outliers_dbscan(df_baseline[col].dropna())
            if not clean_baseline.empty:
                mean_val, std_val = clean_baseline.mean(), clean_baseline.std()
                baseline_stats.append({
                    'AssetID': col, 'Clean_Mean': round(mean_val, 4),
                    'Clean_Std': round(std_val, 4),
                    'Normal_Upper_Limit(3Sigma)': round(mean_val + 3*std_val, 4),
                    'Normal_Lower_Limit(3Sigma)': round(mean_val - 3*std_val, 4),
                    'Outliers_Removed': len(df_baseline[col]) - len(clean_baseline),
                    'Baseline_Data_Count': len(clean_baseline),
                    'Original_Data_Count': len(df_baseline[col].dropna())
                })
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            if ewma_list:
                pd.concat(ewma_list, ignore_index=True).to_excel(writer, sheet_name='EWMA_Comparison', index=False)
            if baseline_stats:
                pd.DataFrame(baseline_stats).to_excel(writer, sheet_name='Baseline_Stats', index=False)
        print(f"✅ ذخیره شد: {output_filename}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_lubrication_4_dual_ewma(file_path, output_filename):
    """تحلیل Dual EWMA سیستم روغن‌کاری (روزانه/هفتگی)"""
    print(f"\n{'='*60}")
    print(f"🔄 [روغن‌کاری-4] Dual EWMA")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9375', 'AssetID_8341', 'AssetID_8343', 'AssetID_8344', 
                      'AssetID_8346', 'AssetID_9286', 'AssetID_9287']
    
    try:
        df = pd.read_excel(file_path, parse_dates=['date'])
        df = df.sort_values('date')
        print(f"✅ دیتا بارگذاری شد. تعداد: {len(df):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False
    
    last_date = df['date'].max()
    one_month_ago = last_date - pd.Timedelta(days=30)
    df_recent = df[df['date'] >= one_month_ago].copy()
    
    alpha_fast, alpha_slow = 0.22, 0.035
    results_list = []
    
    for col in target_sensors:
        if col in df_recent.columns:
            temp_df = df_recent[['date', col]].copy()
            temp_df = temp_df.rename(columns={col: 'Raw_Value'})
            temp_df['AssetID'] = col
            temp_df['Daily_EWMA_Fast'] = temp_df['Raw_Value'].ewm(alpha=alpha_fast, adjust=False).mean()
            temp_df['Weekly_EWMA_Slow'] = temp_df['Raw_Value'].ewm(alpha=alpha_slow, adjust=False).mean()
            temp_df['Signal_Gap'] = temp_df['Daily_EWMA_Fast'] - temp_df['Weekly_EWMA_Slow']
            temp_df['Deviation_Percent'] = (temp_df['Signal_Gap'] / temp_df['Weekly_EWMA_Slow']) * 100
            results_list.append(temp_df[['date', 'AssetID', 'Raw_Value', 'Daily_EWMA_Fast',
                                        'Weekly_EWMA_Slow', 'Signal_Gap', 'Deviation_Percent']])
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            if results_list:
                pd.concat(results_list, ignore_index=True).to_excel(writer, sheet_name='Maintenance_Strategy', index=False)
        print(f"✅ ذخیره شد: {output_filename}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


# ============================================================================
# بخش 2: تعریف وظایف (Jobs) - 3 مجموعه × 4 تحلیل = 12 وظیفه
# ============================================================================

def get_all_jobs():
    """تعریف تمام ۱۲ وظیفه تحلیل - 3 مجموعه × 4 تحلیل"""
    jobs = []
    
    # مجموعه ۱: توربین (Turbine)
    turbine_base = r'second_stage_inputs\G11\dsas_g11_turbine_bearings_output.xlsx'
    turbine_out_base = r'outputs\G11\dsas_g11_bearings_vibration_temp_univariate\univariate\dsas_g11_univariate_output'
    
    jobs.extend([
        {'name': 'Turbine-1: Comprehensive', 'function': analysis_turbine_1_comprehensive,
         'file_path': turbine_base, 'output_filename': f'{turbine_out_base}1.xlsx'},
        {'name': 'Turbine-2: EWMA', 'function': analysis_turbine_2_ewma,
         'file_path': turbine_base, 'output_filename': f'{turbine_out_base}2.xlsx'},
        {'name': 'Turbine-3: EWMA+DBSCAN', 'function': analysis_turbine_3_ewma_dbscan,
         'file_path': turbine_base, 'output_filename': f'{turbine_out_base}3.xlsx'},
        {'name': 'Turbine-4: Dual EWMA', 'function': analysis_turbine_4_dual_ewma,
         'file_path': turbine_base, 'output_filename': f'{turbine_out_base}4.xlsx'}
    ])
    
    # مجموعه ۲: ژنراتور (Generator)
    gen_base = r'second_stage_inputs\G11\dsas_g11_generator_bearings_output.xlsx'
    gen_out_base = r'outputs\G11\dsas_g11_generator_bearings_univariate\univariate\dsas_g11_generator_bearings_univariate_output'
    
    jobs.extend([
        {'name': 'Generator-1: Comprehensive', 'function': analysis_generator_1_comprehensive,
         'file_path': gen_base, 'output_filename': f'{gen_out_base}1.xlsx'},
        {'name': 'Generator-2: EWMA', 'function': analysis_generator_2_ewma,
         'file_path': gen_base, 'output_filename': f'{gen_out_base}2.xlsx'},
        {'name': 'Generator-3: EWMA+DBSCAN', 'function': analysis_generator_3_ewma_dbscan,
         'file_path': gen_base, 'output_filename': f'{gen_out_base}3.xlsx'},
        {'name': 'Generator-4: Dual EWMA', 'function': analysis_generator_4_dual_ewma,
         'file_path': gen_base, 'output_filename': f'{gen_out_base}4.xlsx'}
    ])
    
    # مجموعه ۳: سیستم روغن‌کاری (Lubrication)
    lub_base = r'second_stage_inputs\G11\dsas_g11_lubrication_system_output.xlsx'
    lub_out_base = r'outputs\G11\dsas_g11_lubrication_system_univariate\univariate\dsas_g11_univariate_output'
    
    jobs.extend([
        {'name': 'Lubrication-1: Comprehensive', 'function': analysis_lubrication_1_comprehensive,
         'file_path': lub_base, 'output_filename': f'{lub_out_base}1.xlsx'},
        {'name': 'Lubrication-2: EWMA', 'function': analysis_lubrication_2_ewma,
         'file_path': lub_base, 'output_filename': f'{lub_out_base}2.xlsx'},
        {'name': 'Lubrication-3: EWMA+DBSCAN', 'function': analysis_lubrication_3_ewma_dbscan,
         'file_path': lub_base, 'output_filename': f'{lub_out_base}3.xlsx'},
        {'name': 'Lubrication-4: Dual EWMA', 'function': analysis_lubrication_4_dual_ewma,
         'file_path': lub_base, 'output_filename': f'{lub_out_base}4.xlsx'}
    ])
    
    return jobs


def run_all_analyses():
    """اجرای تمام ۱۲ تحلیل به ترتیب"""
    print("\n" + "="*80)
    print(f"🚀 شروع اجرای همه تحلیل‌ها (۳ مجموعه × ۴ تحلیل = ۱۲ وظیفه)")
    print(f"📅 زمان: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    print("📋 مجموعه‌ها:")
    print("   1. توربین (Turbine) - ۴ تحلیل")
    print("   2. ژنراتور (Generator) - ۴ تحلیل")
    print("   3. سیستم روغن‌کاری (Lubrication) - ۴ تحلیل")
    print("="*80)
    
    jobs = get_all_jobs()
    results = []
    
    for i, job in enumerate(jobs, 1):
        print(f"\n{'#'*80}")
        print(f"# وظیفه {i} از {len(jobs)}: {job['name']}")
        print(f"# ورودی: {job['file_path']}")
        print(f"# خروجی: {job['output_filename']}")
        print(f"{'#'*80}")
        
        try:
            success = job['function'](job['file_path'], job['output_filename'])
            results.append({
                'job_name': job['name'],
                'success': success,
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })
            
            if success:
                print(f"✅ وظیفه {i} با موفقیت کامل شد")
            else:
                print(f"❌ وظیفه {i} با شکست مواجه شد")
                
        except Exception as e:
            print(f"❌ خطای غیرمنتظره در وظیفه {i}: {e}")
            results.append({
                'job_name': job['name'],
                'success': False,
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'error': str(e)
            })
    
    # گزارش نهایی
    print("\n" + "="*80)
    print("📊 گزارش نهایی اجرای همه تحلیل‌ها")
    print("="*80)
    
    success_count = sum(1 for r in results if r['success'])
    total_count = len(results)
    
    print(f"✅ موفق: {success_count} از {total_count}")
    print(f"❌ ناموفق: {total_count - success_count} از {total_count}")
    
    print("\n📋 جزئیات:")
    for i, r in enumerate(results, 1):
        status = "✅" if r['success'] else "❌"
        print(f"   {i:2d}. {status} {r['job_name']} - {r['timestamp']}")
    
    print("="*80)
    print(f"🏁 پایان اجرا در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    
    return results


# ============================================================================
# بخش 3: زمان‌بندی (Scheduler) با دو زمان ۹:۰۰ و ۲۱:۰۰
# ============================================================================

def run_scheduler():
    """بررسی مداوم برای اجرا در زمان‌های مشخص (هر روز ساعت ۹:۰۰ و ۲۱:۰۰)"""
    print("="*80)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد")
    print("📋 شامل: ۳ مجموعه (توربین، ژنراتور، روغن‌کاری) × ۴ تحلیل = ۱۲ خروجی")
    print("="*80)
    print("⏰ زمان‌های اجرا (هر روز):")
    print("   - ساعت 09:00")
    print("   - ساعت 21:00")
    print("="*80)
    print("💡 برای توقف برنامه، Ctrl+C را بزنید")
    print("="*80)
    
    last_run_times = {}
    
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            
            if current_time in ["14:00", "14:10"]:
                if last_run_times.get(current_time) != now.strftime("%Y-%m-%d"):
                    print("\n" + "="*80)
                    print(f"⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                    print("="*80)
                    
                    results = run_all_analyses()
                    last_run_times[current_time] = now.strftime("%Y-%m-%d")
                    
                    print("\n" + "="*80)
                    print("✅ اجرای زمان‌بندی شده با موفقیت کامل شد!")
                    print("="*80)
                    
                    time.sleep(60)
            
            time.sleep(30)
            
        except KeyboardInterrupt:
            print("\n" + "="*80)
            print("⏹️ برنامه با دستور کاربر متوقف شد")
            print(f"⏹️ زمان توقف: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*80)
            break
            
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            time.sleep(60)


# ============================================================================
# بخش 4: اجرای اصلی
# ============================================================================

if __name__ == "__main__":
    try:
        print("="*80)
        print("🚀 شروع برنامه جامع تحلیل (۳ مجموعه یکپارچه)")
        print("="*80)
        print("📋 مجموعه‌ها و خروجی‌ها:")
        print("   ┌─────────────────────────────────────────────────────────┐")
        print("   │  مجموعه ۱: توربین (Turbine)          → ۴ خروجی      │")
        print("   │  مجموعه ۲: ژنراتور (Generator)       → ۴ خروجی      │")
        print("   │  مجموعه ۳: سیستم روغن‌کاری (Lub)     → ۴ خروجی      │")
        print("   └─────────────────────────────────────────────────────────┘")
        print("   مجموع: ۱۲ فایل خروجی")
        print("="*80)
        print("⏰ زمان‌بندی: هر روز ساعت 09:00 و 21:00")
        print("="*80)
        
        run_scheduler()
        
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        import traceback
        traceback.print_exc()
        input("برای خروج Enter بزنید...")

🚀 شروع برنامه جامع تحلیل (۳ مجموعه یکپارچه)
📋 مجموعه‌ها و خروجی‌ها:
   ┌─────────────────────────────────────────────────────────┐
   │  مجموعه ۱: توربین (Turbine)          → ۴ خروجی      │
   │  مجموعه ۲: ژنراتور (Generator)       → ۴ خروجی      │
   │  مجموعه ۳: سیستم روغن‌کاری (Lub)     → ۴ خروجی      │
   └─────────────────────────────────────────────────────────┘
   مجموع: ۱۲ فایل خروجی
⏰ زمان‌بندی: هر روز ساعت 09:00 و 21:00
🔄 برنامه زمان‌بندی خودکار شروع به کار کرد
📋 شامل: ۳ مجموعه (توربین، ژنراتور، روغن‌کاری) × ۴ تحلیل = ۱۲ خروجی
⏰ زمان‌های اجرا (هر روز):
   - ساعت 09:00
   - ساعت 21:00
💡 برای توقف برنامه، Ctrl+C را بزنید
